In [1]:
import pandas as pd
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("features/features_one_week.csv", sep=';')

In [3]:
df_unique_names = df.drop_duplicates(subset='Номенклатура')

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
df_unique_names = df_unique_names.copy()
tfidf = TfidfVectorizer()
tfidf.fit(df_unique_names['Номенклатура'])
text_new = []
for i in range(len(df_unique_names)):
    s = df_unique_names['Номенклатура'].iloc[i]
    df_1 = pd.DataFrame(tfidf.transform([s]).T.todense())
    df_1 = df_1[df_1.values > 0]
    text_new.append(df_1.mean().iloc[0])

df_unique_names['tfidf_mean'] = pd.Series(text_new, index=df_unique_names.index)
df_unique_names.head()

,НеделяНачало,КодТовара,Номенклатура,Папка1,Папка2,Количество,Розничная30%,ЕдиницаИзмерения,ТоварнаяКатегория,Поставщик,...,ДатаПоследнегоПоступления,ОстатокНачалоНедели,temp_mean_week,precip_sum_week,temp_max_week,temp_min_week,НДС,days_off_in_week,sales_lag_2w,tfidf_mean
0,2017-04-10,00-00000030,Биогумус Флорист Бутон 120мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,62.0,шт,Штучный товар,Неизвестный поставщик,...,2017-04-02 12:00:00,10.0,6.142857,7.9,17.2,-5.3,0.18,0,0.0,0.499010
1,2017-04-10,00-00000066,Энерген Аква 10мл. Стимулятор роста растений.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,41.0,шт,Штучный товар,Колхоз,...,2017-04-02 12:00:00,5.0,6.142857,7.9,17.2,-5.3,0.18,0,0.0,0.402379
2,2017-04-10,00-00000058,Фосфатовит универсальный 220мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,125.0,шт,Штучный товар,Неизвестный поставщик,...,2017-04-02 12:00:00,5.0,6.142857,7.9,17.2,-5.3,0.18,0,0.0,0.563057
3,2017-04-10,00-00000052,Удобрение для цитрусовых 285мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,122.5,шт,Штучный товар,"ИП Алексеев Н.С, ПЕНЗА САДОВИТА",...,2017-04-02 12:00:00,2.0,6.142857,7.9,17.2,-5.3,0.18,0,0.0,0.476639
4,2017-04-10,00-00000046,Партенокарпин-био 3мл. Стимулятор плодообразов...,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,83.0,шт,Штучный товар,Неизвестный поставщик,...,2017-04-02 12:00:00,3.0,6.142857,7.9,17.2,-5.3,0.18,0,0.0,0.442425


In [5]:
df = df.merge(
    df_unique_names[['Номенклатура', 'tfidf_mean']],
    on='Номенклатура',
    how='left'
).sort_values(['Номенклатура', 'НеделяНачало']).reset_index(drop=True)

In [6]:
df = df[df['ОстатокНачалоНедели'] % 1 == 0]

In [ ]:
nomenclature_with_supplier = pd.read_csv("features/nomenclature_with_supplier.csv", sep=',')

In [8]:
nomenclature_with_supplier.loc[
    (nomenclature_with_supplier['supplier'] == 'НК') | (nomenclature_with_supplier['supplier'].isna()),
    'supplier'
] = 'Нет значения'

In [9]:
df = df.merge(nomenclature_with_supplier,
              how='left',
              left_on='Номенклатура',
              right_on='product'
             )
df = df.sort_values(['Номенклатура', 'НеделяНачало']).reset_index(drop=True)

In [10]:
df = df.drop(columns=['product'])

In [11]:
df['supplier'] = df['supplier'].fillna('Нет значения')

In [12]:
s = pd.to_datetime(df['ДатаПоследнегоПоступления'], errors='coerce')

df['МесяцПоследнегоПоступления'] = s.dt.month
df['ГодПоследнегоПоступления'] = s.dt.year

df.drop(columns=['ДатаПоследнегоПоступления'], inplace=True)

In [13]:
df = df[df['Количество'] >= 0]
df = df[df['Количество'] <= 350]

In [14]:
df_2 = df[['НеделяНачало', 'Количество', 'КодТовара']]

In [15]:
df_2 = df_2.sort_values(['КодТовара', 'НеделяНачало']).reset_index(drop=True)

df_2_sells = (
    df_2.set_index(['КодТовара', 'НеделяНачало'])[['Количество']]
        .unstack(level=-1)
        .fillna(0)
        .sort_index()
        .sort_index(axis=1)
)

df_2_sells.columns = df_2_sells.columns.get_level_values(1)

In [16]:
from datetime import date, timedelta

df_2_sells.columns = pd.to_datetime(df_2_sells.columns).tz_localize(None)


def get_timespan(df, dt, minus, periods, freq='W-MON'):
    cols = pd.date_range(pd.to_datetime(dt) - timedelta(weeks=minus),
                         periods=periods, freq=freq)
    return df.reindex(columns=cols, fill_value=0)

In [17]:
def prepare_dataset(df, t2017, name_prefix=None):
    X = {}
    # Лаговые фичи по окнам [2, 4, 8, 12] недель
    for i in [2, 4, 8, 12]:
        tmp = get_timespan(df, t2017, i, i)
        X['diff_%s_mean' % i] = tmp.diff(axis=1).mean(axis=1).values
        X['mean_%s_decay' % i] = (tmp * np.power(0.9, np.arange(i)[::-1])).sum(axis=1).values
        X['mean_%s' % i] = tmp.mean(axis=1).values
        X['median_%s' % i] = tmp.median(axis=1).values
        X['min_%s' % i] = tmp.min(axis=1).values
        X['max_%s' % i] = tmp.max(axis=1).values
        X['std_%s' % i] = tmp.std(axis=1).values

    for i in [2, 4, 8, 12]:
        tmp = get_timespan(df, t2017 + timedelta(weeks=-1), i, i)
        X['diff_%s_mean_2' % i] = tmp.diff(axis=1).mean(axis=1).values
        X['mean_%s_decay_2' % i] = (tmp * np.power(0.9, np.arange(i)[::-1])).sum(axis=1).values
        X['mean_%s_2' % i] = tmp.mean(axis=1).values
        X['median_%s_2' % i] = tmp.median(axis=1).values
        X['min_%s_2' % i] = tmp.min(axis=1).values
        X['max_%s_2' % i] = tmp.max(axis=1).values
        X['std_%s_2' % i] = tmp.std(axis=1).values
    X = pd.DataFrame(X)
    
    return X

In [18]:
# Лаговые фичи (с кэшированием в tmp_parts — расчёт долгий)
import pandas as pd
import numpy as np
from pathlib import Path
import gc

df['КодТовара'] = (
    df['КодТовара']
    .astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
)

df['НеделяНачало'] = (
    pd.to_datetime(df['НеделяНачало'], errors='coerce')
      .dt.tz_localize(None).dt.normalize()
      .dt.to_period('W-MON').dt.start_time
)

df_2_sells.index = (
    df_2_sells.index
    .astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
)

df_2_sells.columns = (
    pd.to_datetime(df_2_sells.columns, errors='coerce')
      .tz_localize(None).normalize()
)

TMP_DIR = Path("tmp_parts")
TMP_DIR.mkdir(exist_ok=True)
existing_parts = sorted(TMP_DIR.glob("df_final_part_*.csv"))

if len(existing_parts) == 4:
    print("Кэш tmp_parts найден — пропускаем расчёт лаговых фич")
    parts_paths = existing_parts
else:
    weeks = (
        df['НеделяНачало']
        .dropna().drop_duplicates().sort_values().tolist()
    )
    week_chunks = np.array_split(weeks, 4)
    parts_paths = []

    for i, wk_list in enumerate(week_chunks, start=1):
        wk_list = [w for w in wk_list.tolist() if pd.notna(w)]
        if not wk_list:
            continue
        out_week_frames = []
        for w in wk_list:
            feats = prepare_dataset(df_2_sells, w)
            if isinstance(feats.index, pd.RangeIndex):
                feats.index = df_2_sells.index
            feats.index.name = 'КодТовара'
            feats = feats.reset_index()
            feats['КодТовара'] = (
                feats['КодТовара']
                .astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
            )
            feats['НеделяНачало'] = pd.to_datetime(w).to_datetime64()

            df_week = df.loc[df['НеделяНачало'] == w].copy()
            if df_week.empty:
                del feats; gc.collect(); continue
            df_week['КодТовара'] = (
                df_week['КодТовара']
                .astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
            )
            df_week = df_week.merge(feats, on=['КодТовара', 'НеделяНачало'],
                                    how='left', copy=False)
            out_week_frames.append(df_week)
            del feats, df_week; gc.collect()

        if out_week_frames:
            df_part = pd.concat(out_week_frames, ignore_index=True)
            part_path = TMP_DIR / f"df_final_part_{i}.csv"
            df_part.to_csv(part_path, index=False)
            parts_paths.append(part_path)
            print(f"  Часть {i}/4 готова — {len(df_part)} строк")
            del df_part, out_week_frames; gc.collect()

df = pd.concat(
    [pd.read_csv(p, low_memory=False) for p in parts_paths],
    ignore_index=True
)
print(f"Итого строк после лаговых фич: {len(df)}")

  Часть 1/4 готова — 276255 строк
  Часть 2/4 готова — 437030 строк
  Часть 3/4 готова — 637211 строк
  Часть 4/4 готова — 1120655 строк
Итого строк после лаговых фич: 2471151


In [19]:
# Фильтрация
df = df[(df['Папка1'] != 'разобрать')]
df = df[df['ЕдиницаИзмерения'] == 'шт']
df = df.drop('ЕдиницаИзмерения', axis=1)
df = df[df["ТоварнаяКатегория"] == 'Штучный товар']
df = df.drop('ТоварнаяКатегория', axis=1)

# Проверяем оба столбца на дробные значения
cols = ['ОстатокНачалоНедели', 'Количество']

bad_codes = df.loc[
    (df[cols].astype(float) % 1 != 0).any(axis=1),
    'КодТовара'
].unique()

df = df[~df['КодТовара'].isin(bad_codes)]

df = df[df['Номенклатура'] != '1']

In [20]:
# Train/Test split
df = df[df['НеделяНачало'] > '2021-01-01']

df = df.sort_values('НеделяНачало', ascending=True, kind='mergesort')

X = df.drop('Количество', axis=1).copy()
y = df['Количество'].copy()

X['НеделяНачало'] = pd.to_datetime(X['НеделяНачало'])
X_test, X_train = X[X.НеделяНачало >= '2025-09-02'].copy(), X[X.НеделяНачало < '2025-09-02'].copy()
y_test, y_train = y[y.index.isin(X_test.index)], y[y.index.isin(X_train.index)]

X_test['ТекущийМесяц'] = pd.to_datetime(
    X_test['НеделяНачало'], errors='coerce'
).dt.month
X_test['ТекущийГод'] = pd.to_datetime(
    X_test['НеделяНачало'], errors='coerce'
).dt.year
X_test.drop(columns=['НеделяНачало'], inplace=True)
X_train['ТекущийМесяц'] = pd.to_datetime(
    X_train['НеделяНачало'], errors='coerce'
).dt.month
X_train['ТекущийГод'] = pd.to_datetime(
    X_train['НеделяНачало'], errors='coerce'
).dt.year
X_train.drop(columns=['НеделяНачало'], inplace=True)

X['ТекущийМесяц'] = pd.to_datetime(
    X['НеделяНачало'], errors='coerce'
).dt.month
X['ТекущийГод'] = pd.to_datetime(
    X['НеделяНачало'], errors='coerce'
).dt.year
X.drop(columns=['НеделяНачало'], inplace=True)


object_cols = ['Папка1', 'Папка2', 'Поставщик', 'supplier', 'МесяцПоследнегоПоступления', 'ГодПоследнегоПоступления', 'ТекущийМесяц', 'ТекущийГод']

X[object_cols] = X[object_cols].astype(object)
X_test[object_cols] = X_test[object_cols].astype(object)
X_train[object_cols] = X_train[object_cols].astype(object)

In [21]:
from sklearn.model_selection import TimeSeriesSplit
splitter = TimeSeriesSplit(n_splits=6)


# WAPE
from sklearn.metrics import make_scorer


def wape(y_true, y_pred):
    """WAPE = sum(|y_true - y_pred|) / sum(|y_true|)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom  = np.sum(np.abs(y_true))
    if denom == 0:
        return 0.0
    return np.sum(np.abs(y_true - y_pred)) / denom


wape_scorer = make_scorer(wape, greater_is_better=False)


cols_for_ohe = [x for x in object_cols if X_train[x].nunique() < 5]
cols_for_mte = [x for x in object_cols if X_train[x].nunique() >= 5]
numeric_cols = list(X_train.select_dtypes(exclude='object').columns)

cols_for_ohe_idx = [list(X_train.columns).index(col) for col in cols_for_ohe]
cols_for_mte_idx = [list(X_train.columns).index(col) for col in cols_for_mte]
numeric_cols_idx = [list(X_train.columns).index(col) for col in numeric_cols]

from sklearn.compose import ColumnTransformer
from category_encoders import TargetEncoder
from category_encoders.one_hot import OneHotEncoder
from sklearn.preprocessing import StandardScaler
t = [('OneHotEncoder', OneHotEncoder(), cols_for_ohe_idx),
     ('MeanTargetEncoder', TargetEncoder(), cols_for_mte_idx),
     ('StandardScaler', StandardScaler(), numeric_cols_idx)]
col_transform = ColumnTransformer(transformers=t)
col_transform.fit(X_train, y_train)

from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

pipe = Pipeline([
    ("column_transformer", col_transform),
    ("gradient_boosting", XGBRegressor(
        objective="reg:absoluteerror",
        random_state=42,
        n_estimators=300,
        subsample=1.0,
        min_child_weight=1,
        max_depth=6,
        learning_rate=0.1,
        gamma=1
    ))
])

pipe.fit(X_train, y_train)

Pipeline(steps=[('column_transformer',
                 ColumnTransformer(transformers=[('OneHotEncoder',
                                                  OneHotEncoder(), []),
                                                 ('MeanTargetEncoder',
                                                  TargetEncoder(),
                                                  [2, 3, 5, 16, 17, 18, 75,
                                                   76]),
                                                 ('StandardScaler',
                                                  StandardScaler(),
                                                  [4, 6, 7, 8, 9, 10, 11, 12,
                                                   13, 14, 15, 19, 20, 21, 22,
                                                   23, 24, 25, 26, 27, 28, 29,
                                                   30, 31, 32, 33, 34, 35, 36,
                                                   37, ...])])),
                ('gradient_boosting',
                 XGBRegressor(base_score=None, bo...
                              feature_types=None, feature_weights=None, gamma=1,
                              grow_policy=None, importance_type=None,
                              interaction_constraints=None, learning_rate=0.1,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=6, max_leaves=None, min_child_weight=1,
                              missing=nan, monotone_constraints=None,
                              multi_strategy=None, n_estimators=300,
                              n_jobs=None, num_parallel_tree=None, ...))])

In [22]:
train_preds = pipe.predict(X_train)
test_preds  = pipe.predict(X_test)

train_wape = wape(y_train, train_preds)
test_wape  = wape(y_test,  test_preds)
print('Без CV (гиперпараметры подобраны из головы)')
print(f"WAPE на трейне: {train_wape:.4f}")
print(f"WAPE на тесте:  {test_wape:.4f}")

Без CV (гиперпараметры подобраны из головы)
WAPE на трейне: 0.6820
WAPE на тесте:  0.8315


In [23]:
# GridSearchCV (CV-метрика — WAPE, 4 комбо x 3 фолда = 12 фитов)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

param_grid = {
    "gradient_boosting__n_estimators": [300, 600],
    "gradient_boosting__max_depth":    [4, 6],
}

cv_fast = TimeSeriesSplit(n_splits=3)

search = GridSearchCV(pipe,
                      param_grid,
                      cv=cv_fast,
                      scoring=wape_scorer,
                      n_jobs=1,
                      verbose=3)

search.fit(X_train, y_train)

print(f"Best parameter (CV WAPE={-search.best_score_:.5f}):")
print(search.best_params_)

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
print(f"WAPE лучшей модели на тесте: {wape(y_test, y_pred):.5f}")

Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV 1/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=300;, score=-0.785 total time=  29.0s
[CV 2/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=300;, score=-0.746 total time=  50.9s
[CV 3/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=300;, score=-0.790 total time= 1.3min
[CV 1/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=600;, score=-0.785 total time=  55.0s
[CV 2/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=600;, score=-0.742 total time= 1.6min
[CV 3/3] END gradient_boosting__max_depth=4, gradient_boosting__n_estimators=600;, score=-0.790 total time= 2.5min
[CV 1/3] END gradient_boosting__max_depth=6, gradient_boosting__n_estimators=300;, score=-0.734 total time=  34.5s
[CV 2/3] END gradient_boosting__max_depth=6, gradient_boosting__n_estimators=300;, score=-0.731 total time= 1.0min
[CV 3/3] END gradien

In [24]:
cv_results = pd.DataFrame(search.cv_results_)

fold_cols = [c for c in cv_results.columns
             if c.startswith('split') and c.endswith('_test_score')]
keep_cols = (['param_gradient_boosting__n_estimators',
              'param_gradient_boosting__max_depth']
             + fold_cols
             + ['mean_test_score', 'std_test_score', 'rank_test_score'])

df_cv = cv_results[keep_cols].copy()

for c in fold_cols + ['mean_test_score']:
    df_cv[c] = -df_cv[c]

df_cv = df_cv.rename(columns={
    'param_gradient_boosting__n_estimators': 'n_estimators',
    'param_gradient_boosting__max_depth':    'max_depth',
    'mean_test_score': 'WAPE_mean',
    'std_test_score':  'WAPE_std',
    'rank_test_score': 'rank',
    **{c: f'fold{i+1}_WAPE' for i, c in enumerate(fold_cols)},
})
df_cv = df_cv.sort_values('rank').reset_index(drop=True).round(5)

print("CV-результаты по фолдам (TimeSeriesSplit, n_splits=3):")
print(df_cv.to_string(index=False))
df_cv

CV-результаты по фолдам (TimeSeriesSplit, n_splits=3):
 n_estimators  max_depth  fold1_WAPE  fold2_WAPE  fold3_WAPE  WAPE_mean  WAPE_std  rank
          600          6     0.73449     0.73101     0.73198    0.73249   0.00147     1
          300          6     0.73403     0.73148     0.73456    0.73336   0.00134     2
          600          4     0.78509     0.74237     0.79041    0.77262   0.02150     3
          300          4     0.78509     0.74558     0.79041    0.77369   0.02000     4


,n_estimators,max_depth,fold1_WAPE,fold2_WAPE,fold3_WAPE,WAPE_mean,WAPE_std,rank
0,600,6,0.73449,0.73101,0.73198,0.73249,0.00147,1
1,300,6,0.73403,0.73148,0.73456,0.73336,0.00134,2
2,600,4,0.78509,0.74237,0.79041,0.77262,0.02150,3
3,300,4,0.78509,0.74558,0.79041,0.77369,0.02000,4


In [25]:
# Финальное качество на train и test
final_model = best_model

train_preds_final = final_model.predict(X_train)
test_preds_final  = final_model.predict(X_test)

wape_train_final = wape(y_train, train_preds_final)
wape_test_final  = wape(y_test,  test_preds_final)

print('=' * 60)
print('ФИНАЛЬНОЕ КАЧЕСТВО XGBoost (неделя)')
print('=' * 60)
print(f'  WAPE train: {wape_train_final:.4f}  (n={len(y_train):,})')
print(f'  WAPE test:  {wape_test_final:.4f}  (n={len(y_test):,})')

ФИНАЛЬНОЕ КАЧЕСТВО XGBoost (неделя)
  WAPE train: 0.6686  (n=1,795,199)
  WAPE test:  0.8302  (n=94,105)


In [26]:
# Проверка адекватности модели: R² на train и test
#   R² = 1  — идеальное предсказание;
#   R² = 0  — модель не лучше предсказания среднего;
#   R² < 0  — модель хуже тривиального прогноза.
from sklearn.metrics import r2_score

r2_train = r2_score(y_train, final_model.predict(X_train))
r2_test  = r2_score(y_test,  final_model.predict(X_test))

print(f"R² train: {r2_train:.4f}")
print(f"R² test:  {r2_test:.4f}")

R² train: 0.6947
R² test:  0.5918


In [ ]:
import joblib

save_path = "models/weeks.pkl"
joblib.dump(best_model, save_path)
print(f"Лучшая модель сохранена: {save_path}")

Лучшая модель сохранена: D:\learning_projects\uir_all\models\weeks.pkl
